# 0.2 — Backtest baselines

Sanity-check the walk-forward harness (`fpl/evaluate.py`) and the three baselines (`fpl/modeling/baseline.py`) against real 2025-26 data, and produce the baseline-comparison figure. All feature and metric logic lives in `fpl/`. This notebook only calls it and inspects the result.

In [ ]:
import polars as pl

from fpl.config import PROCESSED_DATA_DIR
from fpl.evaluate import BASELINES, SEASON, gameweek_deadlines, score_baseline, walk_forward
from fpl.modeling.baseline import baseline_minutes_times_ppg
from fpl.plots import baseline_comparison, FIGURES_DIR

player_fixtures = pl.read_parquet(PROCESSED_DATA_DIR / "player_fixtures.parquet").filter(
    pl.col("position") != "AM"
)
player_fixtures.shape

## Walk-forward split sanity check

Deadlines should be the 38 gameweek kickoffs of 2025-26, strictly increasing.

In [ ]:
deadlines = gameweek_deadlines(player_fixtures)
assert len(deadlines) == 38
assert deadlines == sorted(deadlines)
deadlines[:3], deadlines[-3:]

No-leakage check: predicting at gameweek 10's deadline from the full frame must give the same result as predicting from a frame with every later row removed. If a feature ever looked past `asof`, this would fail.

In [ ]:
asof = deadlines[9]
next_asof = deadlines[10]
in_window = (pl.col("kickoff_time") >= asof) & (pl.col("kickoff_time") < next_asof) & (
    pl.col("season") == SEASON
)

full = baseline_minutes_times_ppg(player_fixtures, asof)
truncated = baseline_minutes_times_ppg(
    player_fixtures.filter(pl.col("kickoff_time") < next_asof), asof
)

target_keys = ["code", "GW", "fixture"]
target_full = full.filter(in_window).sort(target_keys)
target_truncated = truncated.filter(in_window).sort(target_keys)

assert target_full.height == target_truncated.height > 0
assert (target_full["xp"] - target_truncated["xp"]).abs().max() < 1e-9
"no leakage detected"

## Baseline metrics

Same walk-forward + scoring `fpl.evaluate.main` uses, run inline for inspection.

In [ ]:
scored = {name: walk_forward(player_fixtures, predict) for name, predict in BASELINES.items()}
summary = pl.DataFrame([score_baseline(name, frame) for name, frame in scored.items()])
summary

`minutes_times_ppg`, predicted minutes times season-to-date points per 90, is the lowest-MAE baseline, consistent with the spec calling it "genuinely hard to beat" (S8, M3). `positional_mean`'s `spearman_60plus` is not comparable to the other two: seed comment in `fpl/evaluate.py`'s generated report for why.

In [ ]:
output_path = FIGURES_DIR / "xp-model" / "1.4-baseline-comparison.png"
baseline_comparison(scored, "mae", output_path)

from IPython.display import Image

Image(filename=str(output_path))